**Visualizing Silver Electrocatalyst Models (Ag 111 and Ag 322)**
This notebook builds, optimizes, and renders silver step geometries and coordination environments for the oxygen reduction reaction.

In [ ]:
!apt install povray
!pip install ase tblite apng

In [ ]:
import os
import numpy as np
from ase import Atoms
from ase.build import bulk, surface, fcc111, add_adsorbate
from ase.io import write, read
from ase.optimize import FIRE
from ase.constraints import FixAtoms
from tblite.ase import TBLite

povray_settings = {
    "display": False,
    "transparent": False,
    "camera_type": "perspective",
    "canvas_width": 1024,
    "camera_dist": 50,
    "area_light": [(0, 0, 100), "White", 200, 200, 1, 1],
}

generic_projection = {
    "rotation": "-75x, 15y, 0z",
    "radii": 1.1,
    "show_unit_cell": 0,
}

def build_ag322():
    ag_bulk = bulk("Ag", "fcc", a=4.09)
    slab = surface(ag_bulk, (3, 2, 2), layers=4, vacuum=10.0)
    slab = slab.repeat((2, 2, 1))
    slab.center()
    fixed_indices = [atom.index for atom in slab if atom.position[2] < np.mean(slab.positions[:, 2])]
    slab.set_constraint(FixAtoms(indices=fixed_indices))
    return slab

model = build_ag322()
add_adsorbate(model, "O", height=1.5, position="fcc")

model.calc = TBLite(method="GFN1-xTB", accuracy=1, max_iterations=50)
optimizer = FIRE(model, trajectory="Ag322_O.traj")
optimizer.run(fmax=0.05)

opt_model = read("Ag322_O.traj", -1)
povray_settings["textures"] = ["intermediate" for _ in opt_model]

write("Ag322_O.pov", opt_model, **generic_projection, povray_settings=povray_settings)
os.system("povray +IAg322_O.pov +Omodel_Ag322_O.png +A +AM2 +UA +Q11 +D")
print("Optimization and rendering complete.")